# Pre-processing Colonies Dataset (2025)

* Data Pre-processing **[done]**
    * Import colonies
    * Import barrier files – reproject all to EPSG 7760
    * Check validity of all shapefiles (turn this into a function…) – also check that all points are in Delhi. (might be part of spatial index notebook and UAC deduplication)    
* Compute barrier clip for all colonies **[done]**
* Run Neighbors Algorithm **[done]**
    * bbox Neighbors algorithm
    * Should check for barriers
    * Should check for NDMC and related areas
* Additional preprocessing for colonies (turn into super function) **[done]**
    * Create index column **[done]**
    * Distance from NDMC **[done]**
    * Area of each polygon **[done]**
* Export GeoDataFrame as joblib file and ESRI Shapefiles

## Import modules and set constants

In [1]:
import os
import joblib
from importlib import reload
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, box
import spatial_index_utils

In [2]:
# WGS 84 / Delhi
epsg_code = 7760

## Import shapefiles **[done]**

In [3]:
colony_filepath = os.path.join('uso_update_sep2021/', 'uso_update_sep2021.shp')

barrier_directory = os.path.join('Barrier_Clip')

canal_filepath = os.path.join(barrier_directory, 'Canal', 'Canal.shp')
drain_filepath = os.path.join(barrier_directory, 'Drain', 'Major_Drain.shp')
railway_filepath = os.path.join(barrier_directory, 'Railway', 'Railway_Line.shp')

# boundary of Delhi
delhi_bounds_filepath = os.path.join('delhi_bounds_buffer', 'delhi_bounds_buffer.shp')

# Check that all filepaths exist
filepath_list = [colony_filepath, canal_filepath, drain_filepath, railway_filepath, delhi_bounds_filepath]

for filepath in filepath_list:
    if not os.path.exists(filepath):
        print('{} does not exist'.format(filepath))

In [4]:
colonies = gpd.read_file(colony_filepath)
canal = gpd.read_file(canal_filepath)
drain = gpd.read_file(drain_filepath)
railway = gpd.read_file(railway_filepath)

## Inspect shapefiles for validity (`check_shapefile`) **[done]**

In [5]:
spatial_index_utils.check_shapefile(gdf=colonies, gdf_name='colonies', 
                                    geom_type='Polygon', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

colonies has duplicate rows: False
----------------------------------------------------
rows with invalid geometries 

----------------------------------------------------
all geometries in colonies are of type Polygon: True
----------------------------------------------------
Rows with None value in geometry column are below
Empty GeoDataFrame
Columns: [AREA, USO_AREA_U, HOUSETAX_C, USO_FINAL, geometry, geom_type]
Index: []
----------------------------------------------------
colonies shapefile is contained within Delhi: True
----------------------------------------------------
Done with shapefile evaluation


In [6]:
spatial_index_utils.check_shapefile(gdf=canal, gdf_name='canal', geom_type='Line', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

canal has duplicate rows: False
----------------------------------------------------
rows with invalid geometries 

----------------------------------------------------
all geometries in canal are of type Line: True
----------------------------------------------------
Rows with None value in geometry column are below
Empty GeoDataFrame
Columns: [FID_1, CAN_NM, CAN_CLSF, EL_GND, DIST_NM, geometry, geom_type]
Index: []
----------------------------------------------------
canal shapefile is contained within Delhi: True
----------------------------------------------------
Done with shapefile evaluation


In [7]:
spatial_index_utils.check_shapefile(gdf=drain, gdf_name='drain', geom_type='Line', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

drain has duplicate rows: False
----------------------------------------------------
rows with invalid geometries 

----------------------------------------------------
all geometries in drain are of type Line: True
----------------------------------------------------
Rows with None value in geometry column are below
Empty GeoDataFrame
Columns: [FID, Drain_type, Drain_Name, MAINTAINED, AC_NAME, DISTRICT, geometry, geom_type]
Index: []
----------------------------------------------------
drain shapefile is contained within Delhi: True
----------------------------------------------------
Done with shapefile evaluation


In [8]:
spatial_index_utils.check_shapefile(gdf=railway, gdf_name='railway', geom_type='Line', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

railway has duplicate rows: False
----------------------------------------------------
rows with invalid geometries 

----------------------------------------------------
all geometries in railway are of type Line: True
----------------------------------------------------
Rows with None value in geometry column are below
Empty GeoDataFrame
Columns: [FID_1, RL_ZONE, geometry, geom_type]
Index: []
----------------------------------------------------
railway shapefile is contained within Delhi: True
----------------------------------------------------
Done with shapefile evaluation


## Remove duplicate geometries **[done]**

In [9]:
# Remove directories
!rm -rf canal.data/
!rm -rf drain.data/
!rm -rf railway.data/
!rm -rf colonies.data/

In [10]:
canal = spatial_index_utils.remove_duplicate_geom(canal)
canal.to_file('canal.data', index=False)

drain = spatial_index_utils.remove_duplicate_geom(drain)
drain.to_file('drain.data', index=False)

railway = spatial_index_utils.remove_duplicate_geom(railway)
railway.to_file('railway.data', index=False)

colonies = spatial_index_utils.remove_duplicate_geom(colonies)
colonies.to_file('colonies.data', index=False)
colonies.to_csv("colonies_no_duplicates_July2025.csv")

43it [00:00, 967.07it/s]


0/43
Original number of rows is 43:
New number of rows after deduplication is: 43


0it [00:00, ?it/s]

0/616


110it [00:01, 75.34it/s]

...

4200/4357


4302it [05:33, 17.92it/s]

4300/4357


4357it [05:37, 12.93it/s]


Original number of rows is 4357:
New number of rows after deduplication is: 4357


In [11]:
len(colonies)

4357

## Import canal, drain, and railway shapefiles

## Check CRS, reproject to EPSG:7760.

In [12]:
colonies.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [13]:
colonies = spatial_index_utils.reproject_gdf(colonies, epsg_code)

GeoDataFrame now has the following CRS:

PROJCRS["WGS 84 / Delhi",BASEGEOGCRS["WGS 84",ENSEMBLE["World Geodetic System 1984 ensemble",MEMBER["World Geodetic System 1984 (Transit)"],MEMBER["World Geodetic System 1984 (G730)"],MEMBER["World Geodetic System 1984 (G873)"],MEMBER["World Geodetic System 1984 (G1150)"],MEMBER["World Geodetic System 1984 (G1674)"],MEMBER["World Geodetic System 1984 (G1762)"],MEMBER["World Geodetic System 1984 (G2139)"],MEMBER["World Geodetic System 1984 (G2296)"],ELLIPSOID["WGS 84",6378137,298.257223563,LENGTHUNIT["metre",1]],ENSEMBLEACCURACY[2.0]],PRIMEM["Greenwich",0,ANGLEUNIT["degree",0.0174532925199433]],ID["EPSG",4326]],CONVERSION["Delhi NSF LCC",METHOD["Lambert Conic Conformal (2SP)",ID["EPSG",9802]],PARAMETER["Latitude of false origin",28.62510126,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8821]],PARAMETER["Longitude of false origin",77,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8822]],PARAMETER["Latitude of 1st standard parallel",28.375,ANG

In [14]:
canal = spatial_index_utils.reproject_gdf(canal, epsg_code)

GeoDataFrame now has the following CRS:

PROJCRS["WGS 84 / Delhi",BASEGEOGCRS["WGS 84",ENSEMBLE["World Geodetic System 1984 ensemble",MEMBER["World Geodetic System 1984 (Transit)"],MEMBER["World Geodetic System 1984 (G730)"],MEMBER["World Geodetic System 1984 (G873)"],MEMBER["World Geodetic System 1984 (G1150)"],MEMBER["World Geodetic System 1984 (G1674)"],MEMBER["World Geodetic System 1984 (G1762)"],MEMBER["World Geodetic System 1984 (G2139)"],MEMBER["World Geodetic System 1984 (G2296)"],ELLIPSOID["WGS 84",6378137,298.257223563,LENGTHUNIT["metre",1]],ENSEMBLEACCURACY[2.0]],PRIMEM["Greenwich",0,ANGLEUNIT["degree",0.0174532925199433]],ID["EPSG",4326]],CONVERSION["Delhi NSF LCC",METHOD["Lambert Conic Conformal (2SP)",ID["EPSG",9802]],PARAMETER["Latitude of false origin",28.62510126,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8821]],PARAMETER["Longitude of false origin",77,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8822]],PARAMETER["Latitude of 1st standard parallel",28.375,ANG

In [15]:
drain = spatial_index_utils.reproject_gdf(drain, epsg_code)

GeoDataFrame now has the following CRS:

PROJCRS["WGS 84 / Delhi",BASEGEOGCRS["WGS 84",ENSEMBLE["World Geodetic System 1984 ensemble",MEMBER["World Geodetic System 1984 (Transit)"],MEMBER["World Geodetic System 1984 (G730)"],MEMBER["World Geodetic System 1984 (G873)"],MEMBER["World Geodetic System 1984 (G1150)"],MEMBER["World Geodetic System 1984 (G1674)"],MEMBER["World Geodetic System 1984 (G1762)"],MEMBER["World Geodetic System 1984 (G2139)"],MEMBER["World Geodetic System 1984 (G2296)"],ELLIPSOID["WGS 84",6378137,298.257223563,LENGTHUNIT["metre",1]],ENSEMBLEACCURACY[2.0]],PRIMEM["Greenwich",0,ANGLEUNIT["degree",0.0174532925199433]],ID["EPSG",4326]],CONVERSION["Delhi NSF LCC",METHOD["Lambert Conic Conformal (2SP)",ID["EPSG",9802]],PARAMETER["Latitude of false origin",28.62510126,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8821]],PARAMETER["Longitude of false origin",77,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8822]],PARAMETER["Latitude of 1st standard parallel",28.375,ANG

In [16]:
railway = spatial_index_utils.reproject_gdf(railway, epsg_code)

GeoDataFrame now has the following CRS:

PROJCRS["WGS 84 / Delhi",BASEGEOGCRS["WGS 84",ENSEMBLE["World Geodetic System 1984 ensemble",MEMBER["World Geodetic System 1984 (Transit)"],MEMBER["World Geodetic System 1984 (G730)"],MEMBER["World Geodetic System 1984 (G873)"],MEMBER["World Geodetic System 1984 (G1150)"],MEMBER["World Geodetic System 1984 (G1674)"],MEMBER["World Geodetic System 1984 (G1762)"],MEMBER["World Geodetic System 1984 (G2139)"],MEMBER["World Geodetic System 1984 (G2296)"],ELLIPSOID["WGS 84",6378137,298.257223563,LENGTHUNIT["metre",1]],ENSEMBLEACCURACY[2.0]],PRIMEM["Greenwich",0,ANGLEUNIT["degree",0.0174532925199433]],ID["EPSG",4326]],CONVERSION["Delhi NSF LCC",METHOD["Lambert Conic Conformal (2SP)",ID["EPSG",9802]],PARAMETER["Latitude of false origin",28.62510126,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8821]],PARAMETER["Longitude of false origin",77,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8822]],PARAMETER["Latitude of 1st standard parallel",28.375,ANG

In [17]:
colonies.crs == drain.crs == canal.crs == railway.crs

True

## Calculate Area (in square kilometers)

In [18]:
colonies['area_km2'] = colonies.area/1000000

In [19]:
colonies.head()

,index,AREA,USO_AREA_U,HOUSETAX_C,USO_FINAL,geometry,geom_type,area_km2
0,0,NEW DELHI 36,5584,None,Planned,"POLYGON Z ((1020282.788 996796.773 0, 1020302....",<property object at 0x7abb9ce4e7f0>,1.966739
1,1,NEW DELHI 35,5585,None,Planned,"POLYGON Z ((1019724.475 994932.797 0, 1019788....",<property object at 0x7abb9ce4e7f0>,0.036429
2,2,NEW DELHI 34,5586,None,Planned,"POLYGON Z ((1019571.955 994876.019 0, 1019571....",<property object at 0x7abb9ce4e7f0>,0.230739
3,3,NEW DELHI 33,5587,None,Planned,"POLYGON Z ((1019352.702 994352.546 0, 1019361....",<property object at 0x7abb9ce4e7f0>,0.281195
4,4,NEW DELHI 32,5588,None,Planned,"POLYGON Z ((1018793.292 994224.182 0, 1018573....",<property object at 0x7abb9ce4e7f0>,0.301253


In [20]:
colonies['area_km2'].max()

np.float64(29.116548965294818)

In [21]:
colonies['area_km2'].min()

np.float64(2.302821695521233e-09)

## Compute barrier clip

In [22]:
colonies.head()

,index,AREA,USO_AREA_U,HOUSETAX_C,USO_FINAL,geometry,geom_type,area_km2
0,0,NEW DELHI 36,5584,None,Planned,"POLYGON Z ((1020282.788 996796.773 0, 1020302....",<property object at 0x7abb9ce4e7f0>,1.966739
1,1,NEW DELHI 35,5585,None,Planned,"POLYGON Z ((1019724.475 994932.797 0, 1019788....",<property object at 0x7abb9ce4e7f0>,0.036429
2,2,NEW DELHI 34,5586,None,Planned,"POLYGON Z ((1019571.955 994876.019 0, 1019571....",<property object at 0x7abb9ce4e7f0>,0.230739
3,3,NEW DELHI 33,5587,None,Planned,"POLYGON Z ((1019352.702 994352.546 0, 1019361....",<property object at 0x7abb9ce4e7f0>,0.281195
4,4,NEW DELHI 32,5588,None,Planned,"POLYGON Z ((1018793.292 994224.182 0, 1018573....",<property object at 0x7abb9ce4e7f0>,0.301253


In [23]:
def remove_select_cols(gdf: gpd.GeoDataFrame):
    candidate_cols = {'index', 'level_0'}
    cols_to_remove = candidate_cols.intersection(gdf.columns)
    return gdf.drop(columns=cols_to_remove)

colonies = remove_select_cols(colonies)
drain = remove_select_cols(drain)
canal = remove_select_cols(canal)
railway = remove_select_cols(railway)

In [24]:
# Note... I had to reset index to make spatial join work!
#colonies = colonies.reset_index()
#colonies = colonies.drop(columns=['index'])

In [25]:
# colonies = gpd.read_file("colonies.data")
# colonies = spatial_index_utils.reproject_gdf(colonies, epsg_code)

In [26]:
colonies.crs == canal.crs

True

In [27]:
colonies.columns

Index(['AREA', 'USO_AREA_U', 'HOUSETAX_C', 'USO_FINAL', 'geometry',
       'geom_type', 'area_km2'],
      dtype='object')

In [28]:
# Create new columns showing intersection with canal, railway and drain
colonies = spatial_index_utils.barrier_intersection(colonies, canal, "canal")

In [29]:
colonies = spatial_index_utils.barrier_intersection(colonies, railway, "railway")

In [30]:
colonies = spatial_index_utils.barrier_intersection(colonies, drain, "drain")

In [31]:
# Create barrier column as being intersection with canal, railway or drain
colonies['barrier'] = colonies['canal'] | colonies['railway'] | colonies["drain"]

In [32]:
colonies.head()

,AREA,USO_AREA_U,HOUSETAX_C,USO_FINAL,geometry,geom_type,area_km2,canal,railway,drain,barrier
0,NEW DELHI 36,5584,None,Planned,"POLYGON Z ((1020282.788 996796.773 0, 1020302....",<property object at 0x7abb9ce4e7f0>,1.966739,False,True,False,True
1,NEW DELHI 35,5585,None,Planned,"POLYGON Z ((1019724.475 994932.797 0, 1019788....",<property object at 0x7abb9ce4e7f0>,0.036429,False,False,False,False
2,NEW DELHI 34,5586,None,Planned,"POLYGON Z ((1019571.955 994876.019 0, 1019571....",<property object at 0x7abb9ce4e7f0>,0.230739,False,False,False,False
3,NEW DELHI 33,5587,None,Planned,"POLYGON Z ((1019352.702 994352.546 0, 1019361....",<property object at 0x7abb9ce4e7f0>,0.281195,False,False,False,False
4,NEW DELHI 32,5588,None,Planned,"POLYGON Z ((1018793.292 994224.182 0, 1018573....",<property object at 0x7abb9ce4e7f0>,0.301253,False,False,False,False


In [33]:
len(colonies)

4357

## Calculate centroid for each polygon

In [34]:
colonies['centroid'] = colonies.centroid
colonies.head()

,AREA,USO_AREA_U,HOUSETAX_C,USO_FINAL,geometry,geom_type,area_km2,canal,railway,drain,barrier,centroid
0,NEW DELHI 36,5584,None,Planned,"POLYGON Z ((1020282.788 996796.773 0, 1020302....",<property object at 0x7abb9ce4e7f0>,1.966739,False,True,False,True,POINT (1020123.175 995898.851)
1,NEW DELHI 35,5585,None,Planned,"POLYGON Z ((1019724.475 994932.797 0, 1019788....",<property object at 0x7abb9ce4e7f0>,0.036429,False,False,False,False,POINT (1019673.024 994869.699)
2,NEW DELHI 34,5586,None,Planned,"POLYGON Z ((1019571.955 994876.019 0, 1019571....",<property object at 0x7abb9ce4e7f0>,0.230739,False,False,False,False,POINT (1019485.484 994565.783)
3,NEW DELHI 33,5587,None,Planned,"POLYGON Z ((1019352.702 994352.546 0, 1019361....",<property object at 0x7abb9ce4e7f0>,0.281195,False,False,False,False,POINT (1019171.868 994576.688)
4,NEW DELHI 32,5588,None,Planned,"POLYGON Z ((1018793.292 994224.182 0, 1018573....",<property object at 0x7abb9ce4e7f0>,0.301253,False,False,False,False,POINT (1018785.675 994590.275)


## Distance from NDMC (turn into function)

In [35]:
# ndmc_center shapefile location
ndmc_center_filepath = os.path.join('ndmc_center7760', 'ndmc_center7760.shp')

# Import shapefile
ndmc_center = gpd.read_file(ndmc_center_filepath)

# Extract NDMC Center as Shapely Point
ndmc_center_point = ndmc_center['geometry'].values[0]

# Code to generate ndmc_distances
# initialize new column with value 0
colonies['ndmc_dist_km'] = 0

# Compute distance from NDMC to centroid of each polygon
# Division by 1000 turns units into kilometers
for idx, row in colonies.iterrows():
    colonies.loc[idx, 'ndmc_dist_km'] = ndmc_center_point.distance(row['centroid'])/1000
    
colonies.head()

/tmp/ipykernel_3832651/3940220929.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '5.159808821021016' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  colonies.loc[idx, 'ndmc_dist_km'] = ndmc_center_point.distance(row['centroid'])/1000


,AREA,USO_AREA_U,HOUSETAX_C,USO_FINAL,geometry,geom_type,area_km2,canal,railway,drain,barrier,centroid,ndmc_dist_km
0,NEW DELHI 36,5584,None,Planned,"POLYGON Z ((1020282.788 996796.773 0, 1020302....",<property object at 0x7abb9ce4e7f0>,1.966739,False,True,False,True,POINT (1020123.175 995898.851),5.159809
1,NEW DELHI 35,5585,None,Planned,"POLYGON Z ((1019724.475 994932.797 0, 1019788....",<property object at 0x7abb9ce4e7f0>,0.036429,False,False,False,False,POINT (1019673.024 994869.699),6.273149
2,NEW DELHI 34,5586,None,Planned,"POLYGON Z ((1019571.955 994876.019 0, 1019571....",<property object at 0x7abb9ce4e7f0>,0.230739,False,False,False,False,POINT (1019485.484 994565.783),6.618792
3,NEW DELHI 33,5587,None,Planned,"POLYGON Z ((1019352.702 994352.546 0, 1019361....",<property object at 0x7abb9ce4e7f0>,0.281195,False,False,False,False,POINT (1019171.868 994576.688),6.709542
4,NEW DELHI 32,5588,None,Planned,"POLYGON Z ((1018793.292 994224.182 0, 1018573....",<property object at 0x7abb9ce4e7f0>,0.301253,False,False,False,False,POINT (1018785.675 994590.275),6.839299


## Create GeoDataFrame with Bounding Box of each Polygon

In [36]:
colonies_bbox = spatial_index_utils.create_bbox_gdf(colonies)

In [37]:
# To generate sindex
colonies_bbox_updated = gpd.GeoDataFrame(colonies_bbox, crs=colonies.crs)

## Spatial Join (intersection of polygon and bbox geometries)

In [38]:
colonies_bbox_nbrs = spatial_index_utils.add_polygon_neighbors_column_fast(polygon_gdf=colonies,
                                                       #right_gdf=colonies_bbox,
                                                       right_gdf = colonies_bbox_updated,
                                                       id_colname='USO_AREA_U', 
                                                       neighbor_colname='nbrs_bbox',
                                                       barrier_colname='barrier')
colonies_bbox_nbrs.head()

100%|███████████████████████████████████████████████████████████████████████████████████| 4357/4357 [00:11<00:00, 387.50it/s]


,AREA,USO_AREA_U,HOUSETAX_C,USO_FINAL,geometry,geom_type,area_km2,canal,railway,drain,barrier,centroid,ndmc_dist_km,nbrs_bbox
0,NEW DELHI 36,5584,None,Planned,"POLYGON Z ((1020282.788 996796.773 0, 1020302....",<property object at 0x7abb9ce4e7f0>,1.966739,False,True,False,True,POINT (1020123.175 995898.851),5.159809,"[3508, 5598, 5599, 3776, 5602, 4011, 3491, 560..."
1,NEW DELHI 35,5585,None,Planned,"POLYGON Z ((1019724.475 994932.797 0, 1019788....",<property object at 0x7abb9ce4e7f0>,0.036429,False,False,False,False,POINT (1019673.024 994869.699),6.273149,"[5594, 5586]"
2,NEW DELHI 34,5586,None,Planned,"POLYGON Z ((1019571.955 994876.019 0, 1019571....",<property object at 0x7abb9ce4e7f0>,0.230739,False,False,False,False,POINT (1019485.484 994565.783),6.618792,"[5596, 5594, 5587, 5585]"
3,NEW DELHI 33,5587,None,Planned,"POLYGON Z ((1019352.702 994352.546 0, 1019361....",<property object at 0x7abb9ce4e7f0>,0.281195,False,False,False,False,POINT (1019171.868 994576.688),6.709542,"[5596, 5586, 5588]"
4,NEW DELHI 32,5588,None,Planned,"POLYGON Z ((1018793.292 994224.182 0, 1018573....",<property object at 0x7abb9ce4e7f0>,0.301253,False,False,False,False,POINT (1018785.675 994590.275),6.839299,"[5596, 5587, 5621, 5620]"


## Calculate neighbor distances

In [39]:
colonies_bbox_nbrs = spatial_index_utils.calc_nbr_dist(polygon_gdf=colonies_bbox_nbrs,
                                  nbr_dist_colname='nbrs_dist_bbox',
                                  centroid_colname='centroid',
                                  neighbor_colname='nbrs_bbox',
                                  neighbor_id_col='USO_AREA_U')

colonies_bbox_nbrs.head()

100%|███████████████████████████████████████████████████████████████████████████████████| 4357/4357 [00:07<00:00, 579.69it/s]


,AREA,USO_AREA_U,HOUSETAX_C,USO_FINAL,geometry,geom_type,area_km2,canal,railway,drain,barrier,centroid,ndmc_dist_km,nbrs_bbox,nbrs_dist_bbox
0,NEW DELHI 36,5584,None,Planned,"POLYGON Z ((1020282.788 996796.773 0, 1020302....",<property object at 0x7abb9ce4e7f0>,1.966739,False,True,False,True,POINT (1020123.175 995898.851),5.159809,"[3508, 5598, 5599, 3776, 5602, 4011, 3491, 560...","[(3508, 0.8833017034389078), (5598, 1.07479036..."
1,NEW DELHI 35,5585,None,Planned,"POLYGON Z ((1019724.475 994932.797 0, 1019788....",<property object at 0x7abb9ce4e7f0>,0.036429,False,False,False,False,POINT (1019673.024 994869.699),6.273149,"[5594, 5586]","[(5594, 0.6299162683013301), (5586, 0.35712228..."
2,NEW DELHI 34,5586,None,Planned,"POLYGON Z ((1019571.955 994876.019 0, 1019571....",<property object at 0x7abb9ce4e7f0>,0.230739,False,False,False,False,POINT (1019485.484 994565.783),6.618792,"[5596, 5594, 5587, 5585]","[(5596, 0.679329044817247), (5594, 0.453264599..."
3,NEW DELHI 33,5587,None,Planned,"POLYGON Z ((1019352.702 994352.546 0, 1019361....",<property object at 0x7abb9ce4e7f0>,0.281195,False,False,False,False,POINT (1019171.868 994576.688),6.709542,"[5596, 5586, 5588]","[(5596, 0.5564745103110775), (5586, 0.31380549..."
4,NEW DELHI 32,5588,None,Planned,"POLYGON Z ((1018793.292 994224.182 0, 1018573....",<property object at 0x7abb9ce4e7f0>,0.301253,False,False,False,False,POINT (1018785.675 994590.275),6.839299,"[5596, 5587, 5621, 5620]","[(5596, 0.6272725729096971), (5587, 0.38643264..."


## Create index column

In [40]:
colonies_bbox_nbrs['index'] = colonies_bbox_nbrs.index

In [41]:
colonies_bbox_nbrs.head()

,AREA,USO_AREA_U,HOUSETAX_C,USO_FINAL,geometry,geom_type,area_km2,canal,railway,drain,barrier,centroid,ndmc_dist_km,nbrs_bbox,nbrs_dist_bbox,index
0,NEW DELHI 36,5584,None,Planned,"POLYGON Z ((1020282.788 996796.773 0, 1020302....",<property object at 0x7abb9ce4e7f0>,1.966739,False,True,False,True,POINT (1020123.175 995898.851),5.159809,"[3508, 5598, 5599, 3776, 5602, 4011, 3491, 560...","[(3508, 0.8833017034389078), (5598, 1.07479036...",0
1,NEW DELHI 35,5585,None,Planned,"POLYGON Z ((1019724.475 994932.797 0, 1019788....",<property object at 0x7abb9ce4e7f0>,0.036429,False,False,False,False,POINT (1019673.024 994869.699),6.273149,"[5594, 5586]","[(5594, 0.6299162683013301), (5586, 0.35712228...",1
2,NEW DELHI 34,5586,None,Planned,"POLYGON Z ((1019571.955 994876.019 0, 1019571....",<property object at 0x7abb9ce4e7f0>,0.230739,False,False,False,False,POINT (1019485.484 994565.783),6.618792,"[5596, 5594, 5587, 5585]","[(5596, 0.679329044817247), (5594, 0.453264599...",2
3,NEW DELHI 33,5587,None,Planned,"POLYGON Z ((1019352.702 994352.546 0, 1019361....",<property object at 0x7abb9ce4e7f0>,0.281195,False,False,False,False,POINT (1019171.868 994576.688),6.709542,"[5596, 5586, 5588]","[(5596, 0.5564745103110775), (5586, 0.31380549...",3
4,NEW DELHI 32,5588,None,Planned,"POLYGON Z ((1018793.292 994224.182 0, 1018573....",<property object at 0x7abb9ce4e7f0>,0.301253,False,False,False,False,POINT (1018785.675 994590.275),6.839299,"[5596, 5587, 5621, 5620]","[(5596, 0.6272725729096971), (5587, 0.38643264...",4


## Remove extraneous columns (`geom_type`)

In [42]:
colonies_bbox_nbrs = colonies_bbox_nbrs.drop(columns=['geom_type']) 

In [43]:
colonies_bbox_nbrs.head()

,AREA,USO_AREA_U,HOUSETAX_C,USO_FINAL,geometry,area_km2,canal,railway,drain,barrier,centroid,ndmc_dist_km,nbrs_bbox,nbrs_dist_bbox,index
0,NEW DELHI 36,5584,None,Planned,"POLYGON Z ((1020282.788 996796.773 0, 1020302....",1.966739,False,True,False,True,POINT (1020123.175 995898.851),5.159809,"[3508, 5598, 5599, 3776, 5602, 4011, 3491, 560...","[(3508, 0.8833017034389078), (5598, 1.07479036...",0
1,NEW DELHI 35,5585,None,Planned,"POLYGON Z ((1019724.475 994932.797 0, 1019788....",0.036429,False,False,False,False,POINT (1019673.024 994869.699),6.273149,"[5594, 5586]","[(5594, 0.6299162683013301), (5586, 0.35712228...",1
2,NEW DELHI 34,5586,None,Planned,"POLYGON Z ((1019571.955 994876.019 0, 1019571....",0.230739,False,False,False,False,POINT (1019485.484 994565.783),6.618792,"[5596, 5594, 5587, 5585]","[(5596, 0.679329044817247), (5594, 0.453264599...",2
3,NEW DELHI 33,5587,None,Planned,"POLYGON Z ((1019352.702 994352.546 0, 1019361....",0.281195,False,False,False,False,POINT (1019171.868 994576.688),6.709542,"[5596, 5586, 5588]","[(5596, 0.5564745103110775), (5586, 0.31380549...",3
4,NEW DELHI 32,5588,None,Planned,"POLYGON Z ((1018793.292 994224.182 0, 1018573....",0.301253,False,False,False,False,POINT (1018785.675 994590.275),6.839299,"[5596, 5587, 5621, 5620]","[(5596, 0.6272725729096971), (5587, 0.38643264...",4


## Save colonies file for Spatial Index

In [44]:
with open('colonies_bbox_nbrs2025.joblib', 'wb') as f:
    joblib.dump(colonies_bbox_nbrs, f)